In [55]:
import os
import numpy as np
from spectral import envi
import json

# Wczytywanie danych z pliku JSON
with open("train_only_labeled.json", "r") as f:
    data = json.load(f)

annotations = data["annotations"]
records = data["records"]
cameras = data["cameras"]
vis_wls = cameras[0]["wavelengths"]
nir_wls = cameras[1]["wavelengths"]

# Mapowanie stanu dojrzałości
ripeness_mapping = {"unripe": 0, "perfect": 1, "overripe": 2}

# Listy na dane i etykiety
X = []
Y = []

# Iteracja po adnotacjach
for annotation in annotations:
    # Znalezienie rekordu powiązanego z adnotacją
    record = next((r for r in records if r["id"] == annotation["record_id"]), None)
    if record:
        try:
            # Wczytywanie obrazu hiperspektralnego
            if record["camera_type"] == "VIS":
                header_file = f"data/{record['files']['header_file']}"
                data_file = f"data/{record['files']['data_file']}"
                image = envi.open(header_file, image=data_file)

                # Załadowanie obrazu jako macierz NumPy
                image_data = image.load()

                # Dodanie danych i etykiet do zbiorów
                X.append(np.array(image_data))
                Y.append(ripeness_mapping[annotation["ripeness_state"]])
        except Exception as e:
            print(f"Error loading image {record['id']}: {e}")

# Konwersja list na macierze NumPy
X = np.array(X, dtype=object)
Y = np.array(Y, dtype=int)

# Informacje o załadowanym zbiorze danych
print(f"Loaded dataset: {len(X)} samples.")

Error loading image 9: Unable to locate file "data/Kiwi/VIS/day_01/kiwi_day_01_03_front.hdr". If the file exists, use its full path or place its directory in the SPECTRAL_DATA environment variable.
Error loading image 11: Unable to locate file "data/Kiwi/VIS/day_01/kiwi_day_01_03_back.hdr". If the file exists, use its full path or place its directory in the SPECTRAL_DATA environment variable.
Error loading image 13: Unable to locate file "data/Kiwi/VIS/day_01/kiwi_day_01_28_front.hdr". If the file exists, use its full path or place its directory in the SPECTRAL_DATA environment variable.
Error loading image 15: Unable to locate file "data/Kiwi/VIS/day_01/kiwi_day_01_28_back.hdr". If the file exists, use its full path or place its directory in the SPECTRAL_DATA environment variable.
Error loading image 25: Unable to locate file "data/Kiwi/VIS/day_02/kiwi_day_02_10_front.hdr". If the file exists, use its full path or place its directory in the SPECTRAL_DATA environment variable.
Error lo

PCA


In [ ]:
import numpy as np
from sklearn.decomposition import PCA

# Docelowy rozmiar (np. 128x128)
target_shape = (128, 128)


def crop_center(image, target_shape):
    """
    Przytnij obraz do określonego rozmiaru z centralnej części.
    """
    h, w = image.shape[:2]
    new_h, new_w = target_shape

    top = max((h - new_h) // 2, 0)
    left = max((w - new_w) // 2, 0)

    cropped_image = image[top : top + new_h, left : left + new_w]
    return cropped_image


# Przycinanie obrazów VIS
cropped_images = []
for sample in X:
    try:
        cropped_image = crop_center(sample, target_shape)
        if cropped_image.shape[:2] == target_shape:
            cropped_images.append(cropped_image)
    except Exception as e:
        print(f"Error cropping image: {e}")

# Spłaszczanie przyciętych obrazów
X_images_flat = np.array([img.reshape(-1) for img in cropped_images])

# Redukcja wymiarów za pomocą PCA
pca = PCA(n_components=50)
X_reduced = pca.fit_transform(X_images_flat)

print(f"Reduced dataset shape: {X_reduced.shape}")

Reduced dataset shape: (92, 50)


In [58]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X_reduced, Y, test_size=0.2, random_state=42
)

SVM


In [61]:
from sklearn.metrics import classification_report
from sklearn.ensemble import AdaBoostClassifier
from sklearn.svm import SVC

clf = SVC(C=1, random_state=42)
clf.fit(X_train, Y_train)

Y_pred = clf.predict(X_test)
print(classification_report(Y_test, Y_pred))

              precision    recall  f1-score   support

           0       1.00      0.25      0.40         4
           1       0.57      0.80      0.67        10
           2       0.50      0.40      0.44         5

    accuracy                           0.58        19
   macro avg       0.69      0.48      0.50        19
weighted avg       0.64      0.58      0.55        19



In [65]:
import numpy as np
import cv2
import pywt
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


# Funkcja do ekstrakcji cech za pomocą FFT
def extract_fft_features(image):
    # Przekształcenie obrazu na dziedzinę częstotliwości
    f = np.fft.fft2(image)
    fshift = np.fft.fftshift(f)  # Przemieszczenie zerowej częstotliwości do centrum
    magnitude = np.abs(fshift)  # Obliczenie magnitudy
    return magnitude.flatten()


# Funkcja do ekstrakcji cech za pomocą DWT
def extract_dwt_features(image, wavelet="db1"):
    # DWT z użyciem falka 'db1'
    coeffs2 = pywt.dwt2(image, wavelet)
    cA, (cH, cV, cD) = coeffs2  # Współczynniki aproksymacji i detali
    return np.concatenate([cA.flatten(), cH.flatten(), cV.flatten(), cD.flatten()])


# Docelowy rozmiar (przycinamy do 128x128)
target_shape = (128, 128)


def crop_center(image, target_shape):
    h, w = image.shape[:2]
    new_h, new_w = target_shape

    top = max((h - new_h) // 2, 0)
    left = max((w - new_w) // 2, 0)

    cropped_image = image[top : top + new_h, left : left + new_w]
    return cropped_image


# Przycinanie i ekstrakcja cech dla obrazu VIS
X_features = []
y = []  # Zawiera etykiety (dojrzałość)

for sample in X:
    image = crop_center(sample, target_shape)
    fft_features = extract_fft_features(image)
    dwt_features = extract_dwt_features(image)

    # Połączenie cech FFT i DWT
    features = np.concatenate([fft_features, dwt_features])

    X_features.append(features)

# Konwertujemy listy na numpy array
X_features = np.array(X_features)
y = np.array(Y)

# Redukcja wymiarów przy pomocy PCA
pca = PCA(n_components=50)
X_reduced = pca.fit_transform(X_features)

# Podział na dane treningowe i testowe
X_train, X_test, y_train, y_test = train_test_split(
    X_reduced, y, test_size=0.2, random_state=42
)

# Testowanie różnych klasyfikatorów

# 1. Klasyfikator SVM
svm_clf = SVC(kernel="linear")
svm_clf.fit(X_train, y_train)
svm_pred = svm_clf.predict(X_test)
svm_accuracy = accuracy_score(y_test, svm_pred)
print(f"SVM Accuracy: {svm_accuracy * 100:.2f}%")

# 2. Klasyfikator Random Forest
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)
rf_pred = rf_clf.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)
print(f"Random Forest Accuracy: {rf_accuracy * 100:.2f}%")

# 3. Klasyfikator KNN
knn_clf = KNeighborsClassifier(n_neighbors=5)
knn_clf.fit(X_train, y_train)
knn_pred = knn_clf.predict(X_test)
knn_accuracy = accuracy_score(y_test, knn_pred)
print(f"KNN Accuracy: {knn_accuracy * 100:.2f}%")

SVM Accuracy: 84.21%
Random Forest Accuracy: 63.16%
KNN Accuracy: 52.63%


Python(60513) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
